# 第18章: グラフニューラルネットワークを最新環境で継続検証する

この Notebook は、読み取り専用の原本 `machine-learning-book/ch18/` を参照しながら、第18章の主要アイデアを `pytest --nbmake` で安定実行できる形に再構成したものです。
PyTorch Geometric への依存は追加せず、グラフ表現、基本 graph convolution、global pooling、スクラッチ GNN によるグラフ分類までをローカル完結で確認します。


## この Notebook で確認すること

- `uv` 環境で Chapter 18 の主要パッケージが利用できることを確認する。
- 原本図版を読み取り専用サブモジュールから参照できることを確認する。
- 隣接行列とノード特徴量からグラフを表現する。
- 基本 graph convolution を行列演算として実装する。
- global sum pooling を使って、可変ノード数のグラフを分類する。


In [ ]:
from importlib.metadata import version
from pathlib import Path
import platform
import random
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Image, display

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

REPO_ROOT = next(
    (
        candidate.resolve()
        for candidate in [Path.cwd(), *Path.cwd().parents]
        if (candidate / 'machine-learning-book').exists()
    ),
    None,
)
assert REPO_ROOT is not None, 'machine-learning-book を含むリポジトリルートを見つけられませんでした'

FIG_DIR = REPO_ROOT / 'machine-learning-book/ch18/figures'
assert FIG_DIR.exists(), f'図版ディレクトリが見つかりません: {FIG_DIR}'

print(f'Python 実行ファイル: {sys.executable}')
print(f'Python バージョン: {platform.python_version()}')
print(f'Matplotlib バックエンド: {matplotlib.get_backend()}')
print(f'図版ディレクトリ: {FIG_DIR}')


In [ ]:
PACKAGE_NAMES = ['numpy', 'pandas', 'matplotlib', 'torch', 'pytest', 'nbmake']
package_versions = pd.DataFrame(
    [(name, version(name)) for name in PACKAGE_NAMES],
    columns=['パッケージ', 'バージョン'],
)
package_versions


## 原本図版の参照

原本の概念図を参照し、グラフ表現、メッセージ伝播、graph pooling の位置づけを確認します。


In [ ]:
selected_figures = [
    ('18_01.png', 420),
    ('18_06.png', 520),
    ('18_09.png', 620),
    ('18_10.png', 520),
]

for figure_name, width in selected_figures:
    figure_path = FIG_DIR / figure_name
    print(figure_path.name)
    display(Image(filename=str(figure_path), width=width))


## グラフを隣接行列とノード特徴量で表現する

原本 Part 1 の小さな色付きグラフを、`networkx` なしで直接テンソルに落とし込みます。
各ノードは色ラベルの one-hot ベクトル、辺は隣接行列で表します。


In [ ]:
color_to_index = {'green': 0, 'blue': 1, 'orange': 2}
node_colors = ['blue', 'orange', 'blue', 'green']
node_features = torch.eye(3)[torch.tensor([color_to_index[color] for color in node_colors])].float()
adjacency = torch.tensor(
    [
        [0., 1., 1., 0.],
        [1., 0., 1., 0.],
        [1., 1., 0., 1.],
        [0., 0., 1., 0.],
    ]
)

adjacency_df = pd.DataFrame(adjacency.numpy(), columns=[f'node_{i}' for i in range(4)])
feature_df = pd.DataFrame(node_features.numpy(), columns=['green', 'blue', 'orange'])

print('隣接行列')
display(adjacency_df)
print('ノード特徴量')
feature_df


## 基本 graph convolution を行列演算として実装する

自己情報と近傍情報を別々の重みで変換し、正規化済み隣接行列で集約します。
これは原本で紹介される基本 graph convolution の最小形です。


In [ ]:
adjacency_hat = adjacency + torch.eye(adjacency.size(0))
degree_inv = torch.diag(1.0 / adjacency_hat.sum(dim=1))
adjacency_norm = degree_inv @ adjacency_hat

weight_self = torch.tensor(
    [[0.2, 0.1, -0.1, 0.3], [0.0, 0.4, 0.2, -0.2], [0.1, -0.3, 0.5, 0.2]],
    dtype=torch.float32,
)
weight_neigh = torch.tensor(
    [[0.3, -0.1, 0.2, 0.0], [0.2, 0.2, -0.2, 0.1], [-0.1, 0.4, 0.1, 0.3]],
    dtype=torch.float32,
)

hidden_features = node_features @ weight_self + adjacency_norm @ node_features @ weight_neigh
pd.DataFrame(hidden_features.numpy(), columns=[f'h_{i}' for i in range(hidden_features.size(1))]).round(3)


## スクラッチ GNN と global sum pooling

次に、基本 graph convolution を PyTorch モジュール化し、複数グラフをまとめて扱うための batch 行列と global sum pooling を実装します。
ラベルは「青ノードが 2 個以上あるグラフかどうか」という小さな分類問題にします。


In [ ]:
class BasicGraphConvolutionLayer(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.weight_self = nn.Parameter(torch.randn(in_channels, out_channels) * 0.2)
        self.weight_neigh = nn.Parameter(torch.randn(in_channels, out_channels) * 0.2)
        self.bias = nn.Parameter(torch.zeros(out_channels))

    def forward(self, X: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
        A_hat = A + torch.eye(A.size(0), device=A.device)
        D_inv = torch.diag(1.0 / A_hat.sum(dim=1))
        A_norm = D_inv @ A_hat
        return X @ self.weight_self + A_norm @ X @ self.weight_neigh + self.bias


def global_sum_pool(X: torch.Tensor, batch_mat: torch.Tensor | None) -> torch.Tensor:
    return torch.sum(X, dim=0, keepdim=True) if batch_mat is None else batch_mat @ X


def get_batch_tensor(graph_sizes: list[int]) -> torch.Tensor:
    total_nodes = sum(graph_sizes)
    batch_mat = torch.zeros(len(graph_sizes), total_nodes)
    start = 0
    for row_idx, size in enumerate(graph_sizes):
        batch_mat[row_idx, start:start + size] = 1.0
        start += size
    return batch_mat


def collate_graphs(graph_batch: list[dict[str, torch.Tensor | int]]) -> dict[str, torch.Tensor]:
    sizes = [graph['A'].size(0) for graph in graph_batch]
    total_nodes = sum(sizes)
    batch_adj = torch.zeros(total_nodes, total_nodes)
    batch_features = torch.cat([graph['X'] for graph in graph_batch], dim=0)
    batch_labels = torch.tensor([graph['y'] for graph in graph_batch])
    batch_mat = get_batch_tensor(sizes)

    offset = 0
    for graph in graph_batch:
        size = graph['A'].size(0)
        batch_adj[offset:offset + size, offset:offset + size] = graph['A']
        offset += size

    return {'A': batch_adj, 'X': batch_features, 'y': batch_labels, 'batch': batch_mat}


def make_graph(colors: list[str], edges: list[tuple[int, int]], label: int) -> dict[str, torch.Tensor | int]:
    X = torch.eye(3)[torch.tensor([color_to_index[color] for color in colors])].float()
    A = torch.zeros(len(colors), len(colors))
    for i, j in edges:
        A[i, j] = 1.0
        A[j, i] = 1.0
    return {'X': X, 'A': A, 'y': label}


graphs = [
    make_graph(['blue', 'blue', 'green'], [(0, 1), (1, 2)], 1),
    make_graph(['blue', 'orange', 'green', 'blue'], [(0, 1), (1, 2), (2, 3)], 1),
    make_graph(['green', 'orange', 'green'], [(0, 1), (1, 2)], 0),
    make_graph(['orange', 'orange', 'green', 'green'], [(0, 1), (1, 2), (2, 3)], 0),
    make_graph(['blue', 'green', 'blue', 'orange'], [(0, 1), (0, 2), (2, 3)], 1),
    make_graph(['green', 'green', 'orange', 'orange'], [(0, 1), (1, 2), (1, 3)], 0),
    make_graph(['blue', 'blue', 'blue', 'green'], [(0, 1), (1, 2), (2, 3)], 1),
    make_graph(['orange', 'green', 'green', 'orange'], [(0, 1), (1, 2), (2, 3)], 0),
]

class NodeNetwork(nn.Module):
    def __init__(self, input_features: int):
        super().__init__()
        self.conv_1 = BasicGraphConvolutionLayer(input_features, 16)
        self.conv_2 = BasicGraphConvolutionLayer(16, 16)
        self.fc_1 = nn.Linear(16, 8)
        self.out_layer = nn.Linear(8, 2)

    def forward(self, X: torch.Tensor, A: torch.Tensor, batch_mat: torch.Tensor | None) -> torch.Tensor:
        x = self.conv_1(X, A).relu()
        x = self.conv_2(x, A).relu()
        x = global_sum_pool(x, batch_mat)
        x = self.fc_1(x).relu()
        return self.out_layer(x)


In [ ]:
graph_batch = collate_graphs(graphs)
model = NodeNetwork(input_features=3)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_history = []

for epoch in range(120):
    optimizer.zero_grad()
    logits = model(graph_batch['X'], graph_batch['A'], graph_batch['batch'])
    loss = F.cross_entropy(logits, graph_batch['y'])
    loss.backward()
    optimizer.step()
    loss_history.append(float(loss.detach()))

with torch.no_grad():
    logits = model(graph_batch['X'], graph_batch['A'], graph_batch['batch'])
    probabilities = logits.softmax(dim=1)
    predictions = probabilities.argmax(dim=1)
    accuracy = (predictions == graph_batch['y']).float().mean().item()

assert accuracy >= 0.99

single_graph_logits = model(graphs[1]['X'], graphs[1]['A'], None)
small_batch = collate_graphs(graphs[:2])
small_batch_logits = model(small_batch['X'], small_batch['A'], small_batch['batch'])
assert torch.allclose(single_graph_logits, small_batch_logits[1:2], atol=1e-5)

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(loss_history)
ax.set_title('スクラッチ GNN の学習損失')
ax.set_xlabel('epoch')
ax.set_ylabel('cross entropy')
ax.grid(alpha=0.3)
plt.show()
plt.close(fig)

pd.DataFrame(
    {
        'graph_id': list(range(len(graphs))),
        'label': graph_batch['y'].tolist(),
        'prediction': predictions.tolist(),
        'probability_blue_rich': probabilities[:, 1].round(decimals=4).tolist(),
    }
)


## まとめ

- 原本 Part 1 のグラフ表現と graph convolution を、追加依存なしでテンソル計算として再構成しました。
- global sum pooling と batch 行列を実装し、複数グラフをまとめて分類できることを確認しました。
- 原本 Part 2 の PyTorch Geometric 例は、CI で継続検証しやすいスクラッチ GNN に置き換えました。
